In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
IdX = np.load("XId_AllStepsR.npy", allow_pickle=True).item()
X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
Y = np.load("Y_AllStepsR.npy")       # class 0 -> 150

In [ ]:
X.shape, len(IdX)

### V7. 1D architecture with ~~triplet~~ contrastive loss and custom-made batches

In [ ]:
# Fixed batch size = 32
N = len(Y)

# this is very crap code, but need to make while with headache...
def batch_generator_contrastive(X, Y, IdX, flat=True):
    pids = np.array(list(IdX.keys()))
    shoes = np.array(list(IdX[1].keys()))
    speeds = np.array(list(IdX[1][0].keys()))

    while True:
        X_ = np.zeros((32, 101, 75, 40), dtype=np.float32)
        Y_ = np.zeros((32, 1), dtype=np.float32)
        
        # 4 x 8 samples
        pid_A = np.random.choice(pids)
        pid_B = np.random.choice(pids)
        pid_C = np.random.choice(pids)
        pid_D = np.random.choice(pids)
        # pid_A, pid_B, pid_C, pid_D = np.random.choice(pids, 4, replace=False)
        
        # 8 samples = 4 same shoe/speed
        shoe_A = np.random.choice(shoes)
        shoe_B = np.random.choice(shoes)
        shoe_C = np.random.choice(shoes)
        shoe_D = np.random.choice(shoes)
        speed_A = np.random.choice(speeds)
        speed_B = np.random.choice(speeds)
        speed_C = np.random.choice(speeds)
        speed_D = np.random.choice(speeds)
        
        # 4 x 4 samples with same shoe/speed
        i = 0
        T = IdX[pid_A][shoe_A][speed_A]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 1
        T = IdX[pid_B][shoe_A][speed_A]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 2
        T = IdX[pid_C][shoe_A][speed_A]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 3
        T = IdX[pid_D][shoe_A][speed_A]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        i = 4
        T = IdX[pid_A][shoe_B][speed_B]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 5
        T = IdX[pid_B][shoe_B][speed_B]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 6
        T = IdX[pid_C][shoe_B][speed_B]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 7
        T = IdX[pid_D][shoe_B][speed_B]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        i = 8
        T = IdX[pid_A][shoe_C][speed_C]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 9
        T = IdX[pid_B][shoe_C][speed_C]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 10
        T = IdX[pid_C][shoe_C][speed_C]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 11
        T = IdX[pid_D][shoe_C][speed_C]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        i = 12
        T = IdX[pid_A][shoe_D][speed_D]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 13
        T = IdX[pid_B][shoe_D][speed_D]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 14
        T = IdX[pid_C][shoe_D][speed_D]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 15
        T = IdX[pid_D][shoe_D][speed_D]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]

        # 4 x 4 samples with random shoe/speed
        i = 16
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_A][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 17
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_B][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 18
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_C][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 19
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_D][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        i = 20
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_A][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 21
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_B][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 22
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_C][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 23
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_D][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        i = 24
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_A][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 25
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_B][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 26
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_C][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 27
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_D][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        i = 28
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_A][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 29
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_B][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 30
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_C][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        i = 31
        shoe = np.random.choice(shoes)
        speed = np.random.choice(speeds)
        T = IdX[pid_D][shoe][speed]
        R = np.random.randint(0, len(T))
        X_[i] = X[T[R]]
        Y_[i] = Y[T[R], 0:1]
        
        if flat:
            X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
        # yield X_, Y_
        yield X_, {"dense": Y_, "lambda": Y_}

gen_train = batch_generator_contrastive(X, Y, IdX)

In [ ]:
A, B = next(gen_train)
A[0].shape

In [ ]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
class SupervisedContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07, name="supcon"):
        super().__init__(name=name)
        self.temperature = temperature

    def call(self, labels, features):
        labels = tf.reshape(labels, [-1])
        labels = tf.cast(labels, tf.int32)

        features = tf.math.l2_normalize(features, axis=1)

        logits = tf.matmul(features, features, transpose_b=True)
        logits = logits / self.temperature

        mask = tf.equal(
            tf.expand_dims(labels, 1),
            tf.expand_dims(labels, 0)
        )

        logits_mask = tf.ones_like(mask, dtype=tf.float32) - tf.eye(tf.shape(labels)[0])
        mask = tf.cast(mask, tf.float32) * logits_mask

        exp_logits = tf.exp(logits) * logits_mask
        log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True) + 1e-9)

        mean_log_prob_pos = tf.reduce_sum(mask * log_prob, axis=1) / (
            tf.reduce_sum(mask, axis=1) + 1e-9
        )

        loss = -tf.reduce_mean(mean_log_prob_pos)
        return loss

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),

    loss={
        "dense": tf.keras.losses.SparseCategoricalCrossentropy(),
        "lambda": SupervisedContrastiveLoss()
    },

    loss_weights={
        "dense": 1.0,
        "lambda": 0.2
    },

    metrics={
        "dense": tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    }
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

class Every100Epochs(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 100 == 0:
            self.model.save(f'checkpoint_epoch_{epoch+1}.keras')
            print(f"\nSaved checkpoint at epoch {epoch+1}")

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(Y)//32//2, # Smaller "fake epochs" for faster feedback
    epochs=500,
    callbacks=[checkpoint, Every100Epochs()]
)

In [ ]:
model.save('last_model.keras')

In [ ]:
model.load_weights('best_model.keras')
model.evaluate(gen_train, steps=len(Y)//32)

In [ ]:
model.load_weights('last_model.keras')
model.evaluate(gen_train, steps=len(Y)//32)